# The data

In [ ]:
from google.colab import auth
import gspread
from google.auth import default
#autenticating to google
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

In [ ]:
import pandas as pd
#defining my worksheet
spreadsheet = gc.open('Cog&MC')
#get_all_values gives a list of rows
#rows = worksheet.get_all_values()
#Convert to a DataFrame
#df = pd.DataFrame(rows)


# Specify the worksheet name
worksheet_name = 'Cog&MC'  # Replace with the actual sheet name

# Open the worksheet by its name
worksheet = spreadsheet.worksheet(worksheet_name)

# Get all values from the worksheet as a list of lists
data = worksheet.get_all_values()

# Create a Pandas DataFrame from the data
df = pd.DataFrame(data)

# Create a DataFrame with the first row as column headers
df = pd.DataFrame(data[1:], columns=data[0])

# Import GLoVe

In [ ]:
pip install transformers


In [ ]:
pip install sentence_transformers

In [ ]:
!wget http://nlp.stanford.edu/data/glove.6B.zip
!unzip glove.6B.zip


# Import ELMo

In [ ]:
import pandas as pd
import numpy as np
import spacy
from tqdm import tqdm
import re
import time
import pickle

In [ ]:
!pip install "tensorflow>=1.7.0"
!pip install tensorflow-hub

In [ ]:
!pip install --upgrade tensorflow-hub

# LR

## GLoVe

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_predict, KFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, mean_squared_error
from sklearn.pipeline import make_pipeline
from sklearn.base import BaseEstimator, TransformerMixin

# Assuming df is your DataFrame
X = df['processed_REF']
y = df['Categorical Metacognition Depth']

class GloveVectorizer(BaseEstimator, TransformerMixin):
    def __init__(self, glove_model):
        self.glove_model = glove_model

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        embeddings = []
        for document in X:
            vectors = [self.glove_model[word] for word in document.split() if word in self.glove_model]
            if vectors:
                document_embedding = np.mean(vectors, axis=0)
            else:
                document_embedding = np.zeros_like(next(iter(self.glove_model.values())))

            embeddings.append(document_embedding)

        return np.vstack(embeddings)

# Load untuned GloVe models
glove_dimensions = [50, 100, 200, 300]
glove_models = {}

for dim in glove_dimensions:
    glove_model_path = f"/content/glove.6B.{dim}d.txt"
    glove_model = {}
    with open(glove_model_path, encoding='utf-8') as f:
        for line in f:
            values = line.split()
            word = values[0]
            vectors = np.asarray(values[1:], dtype='float32')
            glove_model[word] = vectors
    glove_models[dim] = glove_model

# Initialize a list to store the results
results = []

# Initialize a DataFrame to store predictions
predictions_df = pd.DataFrame()

# Iterate over regularization values, dimensions, and penalty types (l1 and l2)
for dim, glove_model in glove_models.items():
    for penalty in ['l1', 'l2']:
        # Creating a pipeline with GloVe and Logistic Regression
        model = make_pipeline(GloveVectorizer(glove_model), LogisticRegression(penalty=penalty, multi_class='multinomial', solver='saga', max_iter=2000))

        # Creating a 10-fold cross-validation object
        kf = KFold(n_splits=10, shuffle=True, random_state=42)

        # Getting cross-validated predictions
        y_pred = cross_val_predict(model, X, y, cv=kf)

        # Append predictions to the DataFrame
        predictions_df[f'{glove_model}_{penalty}'] = y_pred

        # Get the number of iterations from the model
        iterations = model[-1].max_iter

        # Calculating evaluation metrics for LIWC Summary
        weighted_accuracy = accuracy_score(y, y_pred, sample_weight=None)
        mean_squared_error_summary = mean_squared_error(y, y_pred)

        # Calculating MSE for each class
        class_mse = [mean_squared_error(y[y == c], y_pred[y == c]) for c in set(y)]

        # Append results to the list
        results.append({
            'Model': f'GloVe_{dim}d',
            'Penalty': f'LogReg_{penalty}',
            'Weighted Accuracy': f'{weighted_accuracy:.0%}',  # Format as percentage with 2 decimal places
            'MSE': f'{mean_squared_error_summary:.2f}',
            'Class 0 MSE':  f'{class_mse[0]:.2f}',
            'Class 1 MSE':  f'{class_mse[1]:.2f}',
            'Class 2 MSE':  f'{class_mse[2]:.2f}'
        })

        # Clear memory
        #del model, y_pred

# Add actual values to the DataFrame
predictions_df['Actual'] = y.values

# Save the DataFrame with predictions and actual values to a CSV file
from google.colab import files
predictions_df.to_csv('predictions_and_actual.csv')
files.download('predictions_and_actual.csv')

# Create a DataFrame from the results
results_df = pd.DataFrame(results)

# Display the results in a table format
results_df


## ELMo

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, f1_score
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import make_pipeline

import tensorflow as tf
import tensorflow_hub as hub

# Assuming df is your DataFrame
X = df['processed_REF']
y = df['Categorical Metacognition Depth']

# Initialize a DataFrame to store predictions
predictions_df = pd.DataFrame()

# Function to extract ELMo embeddings
def elmo_vectors(x, batch_size=32):
    graph = tf.Graph()
    with graph.as_default():
        elmo = hub.KerasLayer("https://tfhub.dev/google/elmo/2")

    with tf.compat.v1.Session(graph=graph) as sess:
        sess.run(tf.compat.v1.global_variables_initializer())
        sess.run(tf.compat.v1.tables_initializer())

        embeddings = []
        for i in range(0, len(x), batch_size):
            batch = x[i:i+batch_size]
            # Extract ELMo embeddings for the current batch
            batch_embeddings = sess.run(elmo(batch.tolist()))["elmo"]
            embeddings.append(np.mean(batch_embeddings, axis=1))

        # Concatenate embeddings from all batches
        return np.concatenate(embeddings)

# Initialize a list to store the results
results = []

# Creating a 10-fold cross-validation object
kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Iterate over penalty values (L1 and L2)
for penalty in ['l1', 'l2']:
    # Creating a pipeline with ELMo and Logistic Regression
    elmo_model = make_pipeline(FunctionTransformer(elmo_vectors),
                               LogisticRegression(penalty=penalty, multi_class='multinomial', solver='saga', max_iter=2000))

    # Getting cross-validated predictions
    y_pred = cross_val_predict(elmo_model, X, y, cv=kf)

    # Append predictions to the DataFrame
    predictions_df[f'{glove_model}_{penalty}'] = y_pred

    # Get the number of iterations from the model
    iterations = model[-1].max_iter

    # Calculating evaluation metrics
    weighted_accuracy = accuracy_score(y, y_pred, sample_weight=None)
    #class_accuracies = precision_score(y, y_pred, average=None)
    #weighted_f1_score = f1_score(y, y_pred, average='weighted')
    #class_f1_scores = f1_score(y, y_pred, average=None)

    mean_squared_error_summary = mean_squared_error(y, y_pred)

    # Calculating MSE for each class
    class_mse = [mean_squared_error(y[y == c], y_pred[y == c]) for c in set(y)]

    # Append results to the list
    results.append({
        'Model': 'ELMo',
        'Penalty': f'LogReg_{penalty}',
        'Iterations': 2000,
        #'Weighted F1': f'{weighted_f1_score:.2f}',  # Format to 2 decimal places
        #'F1 for class 0': f'{class_f1_scores[0]:.2f}',  # Format to 2 decimal places
        #'F1 for class 1': f'{class_f1_scores[1]:.2f}',  # Format to 2 decimal places
        #'F1 for class 2': f'{class_f1_scores[2]:.2f}',  # Format to 2 decimal places
        'Weighted Accuracy': f'{weighted_accuracy:.0%}',  # Format as percentage with 2 decimal places
        #'Accuracy for class 0': f'{class_accuracies[0]:.0%}',  # Format as percentage with 2 decimal places
        #'Accuracy for class 1': f'{class_accuracies[1]:.0%}',  # Format as percentage with 2 decimal places
        #'Accuracy for class 2': f'{class_accuracies[2]:.0%}',  # Format as percentage with 2 decimal places
        'MSE': f'{mean_squared_error_summary:.2f}',
        'Class 0 MSE':  f'{class_mse[0]:.2f}',
        'Class 1 MSE':  f'{class_mse[1]:.2f}',
        'Class 2 MSE':  f'{class_mse[2]:.2f}'
    })

# Create a DataFrame from the results
results_df = pd.DataFrame(results)

# Display the results
print(results_df)
